# E061 -- NASA Turbofan Engine Degradation (C-MAPSS)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaulVanCode/protoscience-nasa-experiments/blob/main/notebooks/e061_turbofan.ipynb)

**Objective:** Download the NASA C-MAPSS turbofan engine degradation dataset, rank sensors by correlation with engine lifecycle progress, apply SINDy-style sparse regression to discover degradation dynamics, and visualize sensor trajectories over the engine's remaining useful life.

The C-MAPSS (Commercial Modular Aero-Propulsion System Simulation) dataset contains run-to-failure simulations of turbofan engines under different operating conditions and fault modes. Each engine starts healthy and degrades until failure.

## Data Source

- **Dataset:** NASA Prognostics Center of Excellence — Turbofan Engine Degradation Simulation
- **URL:** `https://data.nasa.gov/docs/legacy/CMAPSSData.zip`
- **File:** `train_FD001.txt` — 100 engines, single operating condition, single fault mode
- **Columns:** unit_id, cycle, op_setting_1, op_setting_2, op_setting_3, sensor_1 ... sensor_21
- **License:** Public domain (NASA)

In [ ]:
import urllib.request
import zipfile
import io
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Download C-MAPSS dataset
url = "https://data.nasa.gov/docs/legacy/CMAPSSData.zip"
print("Downloading C-MAPSS dataset from NASA...")
req = urllib.request.Request(url, headers={'User-Agent': 'ProtoScience/1.0'})
response = urllib.request.urlopen(req, timeout=120)
zip_data = io.BytesIO(response.read())
print(f"Downloaded {zip_data.getbuffer().nbytes / 1e6:.1f} MB")

# Extract train_FD001.txt
with zipfile.ZipFile(zip_data) as zf:
    print(f"Files in archive: {zf.namelist()}")
    with zf.open('train_FD001.txt') as f:
        raw = f.read().decode('utf-8')

# Parse space-separated data
# Columns: unit, cycle, op1, op2, op3, s1..s21 (26 total)
lines = [l.strip() for l in raw.strip().split('\n') if l.strip()]
data = np.array([[float(x) for x in line.split()] for line in lines])
print(f"Loaded {data.shape[0]} rows x {data.shape[1]} columns")
print(f"Number of engines: {int(data[:, 0].max())}")

In [ ]:
# Column names
col_names = ['unit', 'cycle', 'op1', 'op2', 'op3'] + [f's{i}' for i in range(1, 22)]

unit = data[:, 0].astype(int)
cycle = data[:, 1].astype(int)
sensors = data[:, 5:]  # 21 sensors
sensor_names = [f's{i}' for i in range(1, 22)]

# Compute Remaining Useful Life (RUL) for each row
# RUL = max_cycle_for_unit - current_cycle
max_cycles = {}
for u in np.unique(unit):
    max_cycles[u] = cycle[unit == u].max()

rul = np.array([max_cycles[u] - c for u, c in zip(unit, cycle)])

# Lifecycle progress: 0 = start, 1 = failure
progress = np.array([c / max_cycles[u] for u, c in zip(unit, cycle)])

print(f"Engine lifespans: min={min(max_cycles.values())}, max={max(max_cycles.values())}, "
      f"mean={np.mean(list(max_cycles.values())):.0f} cycles")
print(f"RUL range: [{rul.min()}, {rul.max()}]")

## Sensor Ranking by Spearman Correlation

We rank all 21 sensors by their Spearman rank correlation with lifecycle progress. Sensors with high |correlation| are the best indicators of degradation — they monotonically change as the engine approaches failure.

In [ ]:
# Rank sensors by Spearman correlation with lifecycle progress
correlations = []
for i in range(21):
    rho, pval = stats.spearmanr(sensors[:, i], progress)
    std = np.std(sensors[:, i])
    correlations.append((sensor_names[i], rho, pval, std))

# Sort by absolute correlation
correlations.sort(key=lambda x: abs(x[1]), reverse=True)

print("=== Sensor Ranking by |Spearman ρ| with Lifecycle Progress ===")
print(f"{'Sensor':>8} {'ρ':>10} {'p-value':>12} {'σ':>10} {'Useful':>8}")
print("-" * 55)
for name, rho, pval, std in correlations:
    useful = '***' if abs(rho) > 0.5 else ('*' if abs(rho) > 0.2 else '-')
    print(f"{name:>8} {rho:>10.4f} {pval:>12.2e} {std:>10.4f} {useful:>8}")

# Top 6 sensors
top6 = [c[0] for c in correlations[:6]]
top6_idx = [sensor_names.index(s) for s in top6]
print(f"\nTop 6 sensors for degradation tracking: {top6}")

## SINDy-style Sparse Regression on Top Sensors

We apply a simplified SINDy (Sparse Identification of Nonlinear Dynamics) approach: for a sample engine, we build a library of candidate functions of the sensor values and their time derivatives, then use thresholded least squares to find sparse governing equations.

This reveals which sensor combinations best predict the rate of degradation.

In [ ]:
# SINDy on a sample engine using top 6 sensors
# Pick engine with median lifespan for representativeness
lifespans = list(max_cycles.values())
median_life = int(np.median(lifespans))
sample_unit = min(max_cycles.keys(), key=lambda u: abs(max_cycles[u] - median_life))
print(f"Sample engine: unit {sample_unit} (lifespan: {max_cycles[sample_unit]} cycles)")

mask = unit == sample_unit
X = sensors[mask][:, top6_idx]  # (T, 6)
t = cycle[mask].astype(float)

# Normalize sensors to [0, 1] for comparability
X_min = X.min(axis=0)
X_max = X.max(axis=0)
X_range = X_max - X_min
X_range[X_range == 0] = 1  # avoid division by zero
X_norm = (X - X_min) / X_range

# Compute derivatives using central differences
dXdt = np.gradient(X_norm, t, axis=0)

# Build SINDy library: [1, x1, x2, ..., x1^2, x1*x2, ...]
n_sensors = X_norm.shape[1]
T = X_norm.shape[0]

library = [np.ones(T)]  # constant
lib_names = ['1']

for i in range(n_sensors):
    library.append(X_norm[:, i])
    lib_names.append(top6[i])

for i in range(n_sensors):
    library.append(X_norm[:, i] ** 2)
    lib_names.append(f"{top6[i]}^2")

for i in range(n_sensors):
    for j in range(i + 1, n_sensors):
        library.append(X_norm[:, i] * X_norm[:, j])
        lib_names.append(f"{top6[i]}*{top6[j]}")

Theta = np.column_stack(library)  # (T, n_features)
print(f"Library shape: {Theta.shape} ({len(lib_names)} features)")

# Thresholded least squares (STLS) for each sensor derivative
threshold = 0.05
n_iter = 10

print(f"\n=== SINDy Equations (threshold={threshold}) ===")
for k in range(n_sensors):
    y = dXdt[:, k]
    xi = np.linalg.lstsq(Theta, y, rcond=None)[0]
    
    for _ in range(n_iter):
        small = np.abs(xi) < threshold
        xi[small] = 0
        big_idx = np.where(~small)[0]
        if len(big_idx) == 0:
            break
        xi[big_idx] = np.linalg.lstsq(Theta[:, big_idx], y, rcond=None)[0]
    
    # Print equation
    terms = []
    for i, (coef, name) in enumerate(zip(xi, lib_names)):
        if abs(coef) > threshold:
            terms.append(f"{coef:+.3f}*{name}")
    if terms:
        eq_str = " ".join(terms)
    else:
        eq_str = "≈ 0 (constant)"
    print(f"  d({top6[k]})/dt = {eq_str}")

In [ ]:
# === 4-SUBPLOT FIGURE ===
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle("E061: NASA Turbofan Engine Degradation (C-MAPSS FD001)",
             fontsize=15, fontweight='bold')

# (a) Sensor correlation ranking
ax = axes[0, 0]
names_sorted = [c[0] for c in correlations]
rhos_sorted = [c[1] for c in correlations]
colors = ['crimson' if abs(r) > 0.5 else ('orange' if abs(r) > 0.2 else 'gray')
          for r in rhos_sorted]
ax.barh(range(21), rhos_sorted, color=colors, edgecolor='black', linewidth=0.3)
ax.set_yticks(range(21))
ax.set_yticklabels(names_sorted, fontsize=8)
ax.set_xlabel('Spearman ρ with lifecycle progress', fontsize=11)
ax.set_title('(a) Sensor Degradation Correlation Ranking', fontsize=12)
ax.axvline(0, color='black', linewidth=0.5)
ax.invert_yaxis()

# (b) Degradation curves for top 6 sensors (sample engine)
ax = axes[0, 1]
prog_sample = progress[mask]
for k in range(6):
    ax.plot(prog_sample, X_norm[:, k], label=top6[k], alpha=0.8, linewidth=1.5)
ax.set_xlabel('Lifecycle Progress (0=new, 1=failure)', fontsize=11)
ax.set_ylabel('Normalized Sensor Value', fontsize=11)
ax.set_title(f'(b) Degradation Curves — Engine {sample_unit}', fontsize=12)
ax.legend(fontsize=9, ncol=2)
ax.set_xlim(0, 1)

# (c) Multi-engine overlay for best sensor
ax = axes[1, 0]
best_sensor_idx = sensor_names.index(correlations[0][0])
best_name = correlations[0][0]
cmap = plt.cm.viridis
engine_ids = np.unique(unit)
for i, uid in enumerate(engine_ids[:30]):  # Show 30 engines
    m = unit == uid
    p = progress[m]
    s = sensors[m, best_sensor_idx]
    ax.plot(p, s, color=cmap(i / 30), alpha=0.4, linewidth=0.8)
ax.set_xlabel('Lifecycle Progress', fontsize=11)
ax.set_ylabel(f'{best_name} (raw units)', fontsize=11)
ax.set_title(f'(c) {best_name} Degradation — 30 Engines Overlaid', fontsize=12)

# (d) Lifespan distribution
ax = axes[1, 1]
ax.hist(lifespans, bins=20, color='steelblue', edgecolor='black', linewidth=0.5, alpha=0.8)
ax.axvline(np.mean(lifespans), color='red', linestyle='--', linewidth=2,
           label=f'Mean = {np.mean(lifespans):.0f} cycles')
ax.axvline(np.median(lifespans), color='orange', linestyle='--', linewidth=2,
           label=f'Median = {np.median(lifespans):.0f} cycles')
ax.set_xlabel('Engine Lifespan (cycles)', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('(d) Engine Lifespan Distribution (100 engines)', fontsize=12)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('e061_turbofan.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nFigure saved: e061_turbofan.png")

## Key Results Summary

| Analysis | Result |
|----------|--------|
| Top degradation sensors | Identified by Spearman |ρ| > 0.5 |
| SINDy equations | Sparse governing dynamics for top sensors |
| Degradation pattern | Monotonic sensor drift as engine approaches failure |
| Lifespan variability | Significant spread across 100 engines |

**Physical interpretation:** Turbofan engines degrade through multiple coupled mechanisms (blade erosion, combustion chamber fouling, seal leakage). The top-ranked sensors capture these processes and could feed a predictive maintenance model. The SINDy equations reveal which sensor interactions drive the degradation dynamics.